Neha Pralhad Lahane


In [9]:
!pip install requests beautifulsoup4 pandas lxml deepface tf-keras fastapi uvicorn python-multipart pyngrok opencv-python-headless

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
]

def get_headers():
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-IN,en-GB;q=0.9,en-US;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "DNT": "1",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Referer": "https://www.amazon.in/",
    }

def create_session():
    session = requests.Session()
    print("🔗 Creating session with Amazon homepage...")
    try:
        session.get("https://www.amazon.in", headers=get_headers(), timeout=15)
        print(" Session created with cookies")
        time.sleep(random.uniform(2, 4))
    except Exception as e:
        print(f"Warning: {e}")
    return session

def fetch_page(session, url, max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(url, headers=get_headers(), timeout=20)
            if response.status_code == 200:
                if "Enter the characters you see below" in response.text:
                    print(f"  CAPTCHA on attempt {attempt}")
                    time.sleep(random.uniform(5, 10))
                    continue
                return response
            elif response.status_code == 503:
                wait = random.uniform(5, 10) * attempt
                print(f"   503 error - attempt {attempt}/{max_retries}, waiting {wait:.0f}s...")
                time.sleep(wait)
            else:
                print(f"   HTTP {response.status_code}")
                time.sleep(3)
        except Exception as e:
            print(f"   Error: {e}")
            time.sleep(5)
    return None

def parse_products(html_content, page_number):
    soup = BeautifulSoup(html_content, "lxml")
    cards = soup.find_all("div", {"data-component-type": "s-search-result"})
    print(f"    Found {len(cards)} products on page {page_number}")
    products = []
    for card in cards:
        title_tag   = card.find("h2")
        price_tag   = card.find("span", class_="a-price-whole")
        rating_tag  = card.find("span", class_="a-icon-alt")
        img_tag     = card.find("img", class_="s-image")
        link_tag    = card.find("a", class_="a-link-normal s-no-outline")
        if not link_tag and card.find("h2"):
            link_tag = card.find("h2").find("a")
        sponsored = card.find(
            lambda t: t.name in ["span","div"] and t.string and "Sponsored" in t.string
        )
        products.append({
            "Title":      title_tag.get_text(strip=True) if title_tag else "N/A",
            "Price":      "₹" + price_tag.get_text(strip=True).replace(",","") if price_tag else "N/A",
            "Rating":     rating_tag.get_text(strip=True) if rating_tag else "N/A",
            "Image URL":  img_tag["src"] if img_tag else "N/A",
            "Product URL":"https://www.amazon.in" + link_tag["href"] if link_tag else "N/A",
            "Ad/Organic": "Ad (Sponsored)" if sponsored else "Organic",
            "Page":       page_number,
        })
    return products

def scrape_amazon_laptops(total_pages=3):
    print("="*55)
    print("  Amazon India - Laptop Scraper")
    print("="*55)
    session = create_session()
    all_products = []
    for page in range(1, total_pages + 1):
        url = f"https://www.amazon.in/s?k=laptops&page={page}"
        print(f"\n📄 Scraping page {page}/{total_pages}...")
        response = fetch_page(session, url)
        if response is None:
            print(f"  Failed page {page}, skipping.")
            continue
        products = parse_products(response.content, page)
        all_products.extend(products)
        if page < total_pages:
            delay = random.uniform(4, 8)
            print(f"   ⏳ Waiting {delay:.1f}s...")
            time.sleep(delay)
    return pd.DataFrame(all_products)

# ▶️ RUN SCRAPER
df = scrape_amazon_laptops(total_pages=3)
print(f"\n🎉 Total products: {len(df)}")

  Amazon India - Laptop Scraper
🔗 Creating session with Amazon homepage...
 Session created with cookies

📄 Scraping page 1/3...
    Found 16 products on page 1
   ⏳ Waiting 4.4s...

📄 Scraping page 2/3...
    Found 16 products on page 2
   ⏳ Waiting 4.1s...

📄 Scraping page 3/3...
    Found 16 products on page 3

🎉 Total products: 48


In [11]:
# Show the data as a table
df.head(20)

,Title,Price,Rating,Image URL,Product URL,Ad/Organic,Page
0,"Acer Smartchoice Aspire One, AMD Ryzen 3-7320U...",₹39990,5.0 out of 5 stars,https://m.media-amazon.com/images/I/71ouu-iX3p...,https://www.amazon.in/Acer-Smartchoice-3-7320U...,Organic,1
1,𝗗𝗲𝗹𝗹Laptop Model 5420 | 𝗜𝗡𝗧𝗘𝗟i5 11th Gen Proce...,₹32490,5.0 out of 5 stars,https://m.media-amazon.com/images/I/51LHNjAjH9...,https://www.amazon.in/%F0%9D%97%97%F0%9D%97%B2...,Organic,1
2,"ASUS TUF A14 (2026),AMD Ryzen AI MAX+ 392, AMD...",₹219990,N/A,https://m.media-amazon.com/images/I/81OqrEm5FS...,https://www.amazon.in/ASUS-Radeon-Windows-FA40...,Organic,1
3,"Dell 15 (Previously Inspiron) Laptop, 14th Gen...",₹44990,4.1 out of 5 stars,https://m.media-amazon.com/images/I/717WZ7Wriw...,https://www.amazon.in/Dell-Previously-Inspiron...,Organic,1
4,Lenovo V15 G4 AMD Athlon Silver 7120U Laptop 8...,₹42999,4.0 out of 5 stars,https://m.media-amazon.com/images/I/61AccNkmFF...,https://www.amazon.in/Lenovo-V15-Lifetime-Vali...,Organic,1
5,"ASUS Chromebook CX1405 (2026), Smartchoice,Int...",₹30990,2.7 out of 5 stars,https://m.media-amazon.com/images/I/616Jpqwdp3...,https://www.amazon.in/ASUS-Chromebook-Smartcho...,Organic,1
6,"ASUS Vivobook 15 (2026),Intel Core 3 100U (14t...",₹53990,N/A,https://m.media-amazon.com/images/I/71i8u4V18Z...,https://www.amazon.in/ASUS-Vivobook-Windows-Of...,Organic,1
7,"Acer Smartchoice Aspire One, Intel Core Celero...",₹38990,3.3 out of 5 stars,https://m.media-amazon.com/images/I/71uoRYyilh...,https://www.amazon.in/Acer-Smartchoice-Celeron...,Organic,1
8,"HP 15 (2026), AMD Ryzen 3 Quad Core 7335U - (8...",₹43640,N/A,https://m.media-amazon.com/images/I/71vF+WK+vr...,https://www.amazon.in/HP-15-Ryzen-Quad-7335U/d...,Organic,1
9,Lenovo IdeaPad Slim 3 12th Gen Intel Core i5-1...,₹61890,4.5 out of 5 stars,https://m.media-amazon.com/images/I/61kNhhiAxS...,https://www.amazon.in/Lenovo-IdeaPad-i5-12450H...,Organic,1


In [12]:
from google.colab import files
import glob

# Find the CSV file
csv_files = glob.glob("amazon_laptops_*.csv")

if csv_files:
    filename = csv_files[0]
    print(f"Found: {filename}")
    files.download(filename)  # Auto downloads to your Mac
else:
    print("CSV not found, running scraper again...")

Found: amazon_laptops_20260605_073734.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
print("Summary:")
print(f"  Total products : {len(df)}")
print(f"  Ads found      : {len(df[df['Ad/Organic']=='Ad (Sponsored)'])}")
print(f"  Organic results: {len(df[df['Ad/Organic']=='Organic'])}")
print(f"  With price     : {len(df[df['Price']!='N/A'])}")
print(f"  With rating    : {len(df[df['Rating']!='N/A'])}")
print(f"\n  Columns: {list(df.columns)}")

Summary:
  Total products : 48
  Ads found      : 0
  Organic results: 48
  With price     : 48
  With rating    : 39

  Columns: ['Title', 'Price', 'Rating', 'Image URL', 'Product URL', 'Ad/Organic', 'Page']
